In [ ]:
!pip install zipfile
!pip install pypdf
!pip install dotenv
!pip install pinecone
!pip install langchain_community

ERROR: Could not find a version that satisfies the requirement zipfile (from versions: none)

[notice] A new release of pip is available: 25.0.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for zipfile


In [27]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv(), override=True)

PINECONE_API_KEY  = os.getenv('PINECONE_API_KEY')
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [ ]:
import zipfile

ZIP_FILE_DIR = "./documents"
ZIP_FILE_NAME = "documents.zip"

with zipfile.ZipFile(ZIP_FILE_DIR + "/" + ZIP_FILE_NAME, 'r') as referencia_zip:
    referencia_zip.extractall(ZIP_FILE_DIR)

In [8]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("./documents/documents/internal_docs_by_area/Customer_support/IronStore_Customer_Support_Escalation_Procedures.pdf")

pages = loader.load()

print(pages[0].page_content)

Customer Support Escalation Procedures
Owning area: Customer_support
Last reviewed: August 2026
Document ID: CS-POL-014
Version: 3.2
Applies to: IronStore customer support operations across Europe
1. Purpose and Overview
This procedure defines how IronStore identifies, manages, and escalates customer contacts that cannot be resolved
through standard frontline support. It is intended to ensure consistent decisions, timely ownership, and appropriate
protection of customer, payment, product, and company information.
Escalation is required when a case involves material customer impact, operational risk, legal or regulatory
considerations, reputational risk, or a resolution outside the agent’s authority. Escalation does not remove ownership
from the original agent unless a receiving team formally accepts the case.
The procedure applies to contacts received through email, telephone, chat, social media, marketplace messaging,
and the IronStore Help Centre.
2. Scope
This procedure covers:
  O

In [ ]:
from pinecone import Pinecone

pc = Pinecone(api_key=PINECONE_API_KEY)

In [ ]:
import re
from pathlib import Path
from langchain_core.documents import Document

def load_clean_document_content(pages):
    page_lines = []
    document_content = ""

    for page in pages:
        page_lines = [re.sub(r'\x7f|Page\s*\d+\s*of\s*\d+\s*—', "", line).strip() for line in page.page_content.split("\n") if line.strip()]
        if len(page_lines):
            document_content += " ".join(page_lines)
    return document_content

def get_documents_list_content(base_path):
    documents_list = []
    
    for file in base_path.rglob("*"):
        if file.is_file():
            file_name = file.name;
            if file_name != '.DS_Store':
                loader = PyPDFLoader(file.as_posix())
                documents_list.append(
                    Document(
                        page_content=load_clean_document_content(loader.load()),
                        metadata={
                            "heading": file_name,
                            "file_name": file_name,
                            "file_path": file.relative_to(base_path).as_posix()
                        }
                    )
                )
            else:
                file.unlink()
    return documents_list

In [ ]:
from pinecone import ServerlessSpec
from langchain_pinecone import PineconeVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter

PINECONE_INDEX_NAME = "enterprise-ai-assistant"
embeddings = OpenAIEmbeddings(api_key=OPENAI_API_KEY)
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)

# Create new pinecone index
if PINECONE_API_KEY not in pc.list_indexes().names():
    pc.create_index(
        name=PINECONE_INDEX_NAME,
        dimension=1536, # Standard dimensions for OpenAI embeddings
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
    
documents_list = get_documents_list_content(Path("./documents/documents"))

for document in documents_list:
    chunks = text_splitter.split_documents(document)
    if chunks:
        PineconeVectorStore.from_documents(
            documents=chunks,
            embedding=embeddings,
            index_name=PINECONE_INDEX_NAME,
            namespace=document.metadata.get("file_name", "unknown")
        )